# Fire Risk YOLO Training on Google Colab

This notebook trains the single-class `fire_risk` detector. It expects the project files plus the dataset under `datasets/`, or a dataset zip in Google Drive.

## 1. Runtime

In Colab, choose **Runtime -> Change runtime type -> T4 GPU** or another GPU before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, zipfile
from pathlib import Path

# If you already cloned/uploaded the repo into /content/fire_detection, leave this as-is.
PROJECT_DIR = Path('/content/fire_detection')

# Optional: if your repo is on GitHub, set REPO_URL and uncomment the clone block.
REPO_URL = ''  # e.g. 'https://github.com/USER/fire_detection.git'

if not PROJECT_DIR.exists() and REPO_URL:
    !git clone {REPO_URL} {PROJECT_DIR}

if not PROJECT_DIR.exists():
    raise RuntimeError('Upload/clone the project to /content/fire_detection or set REPO_URL.')

os.chdir(PROJECT_DIR)
print('Project:', Path.cwd())

## 2. Install Dependencies

In [ ]:
!python -m pip install -U pip
!pip install -r requirements.txt
# Match the local training runtime if available; this is safe to skip if pip resolves newer compatible packages.
!pip install 'ultralytics==8.4.60'

import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 3. Prepare Dataset

Because `datasets/` is gitignored, put a zip at `DATA_ZIP` if the dataset is not already inside the project. The zip should contain either `datasets/mixed_indoor_fire_risk_plus_new/...` or `datasets/mixed_indoor_fire_plus_new/...`.

In [ ]:
DATA_ZIP = Path('/content/drive/MyDrive/fire_detection_datasets.zip')

if not Path('datasets/mixed_indoor_fire_risk_plus_new/data.yaml').exists():
    if DATA_ZIP.exists():
        print('Unzipping dataset:', DATA_ZIP)
        with zipfile.ZipFile(DATA_ZIP) as zf:
            zf.extractall(PROJECT_DIR)
    else:
        print('No dataset zip found at', DATA_ZIP)

if not Path('datasets/mixed_indoor_fire_risk_plus_new/data.yaml').exists():
    !python scripts/prepare_fire_risk_dataset.py --source datasets/mixed_indoor_fire_plus_new --output datasets/mixed_indoor_fire_risk_plus_new --copy-images

!cat datasets/mixed_indoor_fire_risk_plus_new/data.yaml

## 4. Train Experiment A

Default: single-class `fire_risk`, `imgsz=768`, `60` epochs. Results are copied to Drive when training finishes.

In [ ]:
DRIVE_OUTPUT = '/content/drive/MyDrive/fire_detection_colab_runs'
!python scripts/train_fire_risk_colab.py \
  --data datasets/mixed_indoor_fire_risk_plus_new/data.yaml \
  --epochs 60 \
  --imgsz 768 \
  --batch 8 \
  --device 0 \
  --name yolo26n_fire_risk_colab_768_60e \
  --drive-output {DRIVE_OUTPUT}

## 5. Summarize Metrics

In [ ]:
import csv
from pathlib import Path

run_dir = Path('runs/train/yolo26n_fire_risk_colab_768_60e')
results_csv = run_dir / 'results.csv'
rows = list(csv.DictReader(results_csv.open()))
best = max(rows, key=lambda r: float(r['metrics/mAP50(B)']))
print('best_epoch:', best['epoch'])
print('precision:', best['metrics/precision(B)'])
print('recall:', best['metrics/recall(B)'])
print('mAP50:', best['metrics/mAP50(B)'])
print('mAP50-95:', best['metrics/mAP50-95(B)'])
print('best weight:', run_dir / 'weights/best.pt')